# Atividade 2 — Camada Bronze

Cria do banco de dados, se não existir, e as tabelas a partir dos CSVs fornecidos, realiza a leitura dos dados sem alterações no conteúdo e adiciona uma coluna como o momento de inserção.

## 1. Preparar o banco de dados

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")


## 2. Definir os arquivos de origem

In [0]:
caminho_dados = "/Volumes/workspace/movies/dados_movies/"

fontes_csv = {
    "movies_info_IMDB_TMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews",
}


## 3. Ingerir os CSVs sem transformação

In [0]:
from pyspark.sql import functions as F

opcoes_csv = {
    "header": "true",
    "multiLine": "true",
    "escape": '"',
}

def carregar_csv_na_bronze(nome_arquivo: str, nome_tabela: str) -> None:
    origem = caminho_dados.rstrip("/") + "/" + nome_arquivo

    dados_brutos = (
        spark.read
        .options(**opcoes_csv)
        .csv(origem)
    )

    resultado = dados_brutos.withColumn(
        "ingestion_datetime", F.current_timestamp()
    )

    (resultado.write
        .format("delta")
        .mode("append")
        .saveAsTable(nome_tabela))

for arquivo, tabela in fontes_csv.items():
    carregar_csv_na_bronze(arquivo, tabela)


## 4. Configurar o período da API PTAX

In [0]:
from datetime import date, datetime, timedelta

# Cria os widgets somente quando ainda não existem
for nome_widget in ("data_inicio", "data_fim"):
    try:
        dbutils.widgets.get(nome_widget)
    except Exception:
        dbutils.widgets.text(nome_widget, "")

data_fim_texto = dbutils.widgets.get("data_fim").strip()
data_inicio_texto = dbutils.widgets.get("data_inicio").strip()

if not data_fim_texto:
    data_fim_texto = date.today().strftime("%m-%d-%Y")
if not data_inicio_texto:
    data_inicio_texto = (date.today() - timedelta(days=7)).strftime("%m-%d-%Y")

# Valida o padrão solicitado pela API e a ordem cronológica
data_inicio_obj = datetime.strptime(data_inicio_texto, "%m-%d-%Y").date()
data_fim_obj = datetime.strptime(data_fim_texto, "%m-%d-%Y").date()
if data_inicio_obj > data_fim_obj:
    raise ValueError("data_inicio deve ser anterior ou igual a data_fim.")

print(f"Período solicitado: {data_inicio_texto} a {data_fim_texto}")


## 5. Consultar a cotação de compra do dólar

In [0]:
import requests

url_ptax = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,"
    "dataFinalCotacao=@dataFinalCotacao)"
)
parametros_ptax = {
    "@dataInicial": f"'{data_inicio_texto}'",
    "@dataFinalCotacao": f"'{data_fim_texto}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json",
}

resposta = requests.get(url_ptax, params=parametros_ptax, timeout=60)
resposta.raise_for_status()
registros_ptax = resposta.json().get("value", [])

print(f"Cotações recebidas: {len(registros_ptax)}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5718433261581126>, line 9
      1 import requests
      3 url_ptax = (
      4     "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
      5     "CotacaoDolarPeriodo(dataInicial=@dataInicial,"
      6     "dataFinalCotacao=@dataFinalCotacao)"
      7 )
      8 parametros_ptax = {
----> 9     "@dataInicial": f"'{data_inicio_texto}'",
     10     "@dataFinalCotacao": f"'{data_fim_texto}'",
     11     "$select": "dataHoraCotacao,cotacaoCompra",
     12     "$format": "json",
     13 }
     15 resposta = requests.get(url_ptax, params=parametros_ptax, timeout=60)
     16 resposta.raise_for_status()

NameError: name 'data_inicio_texto' is not defined

## 6. Gravar a resposta da API na Bronze


In [0]:
from pyspark.sql.types import StructType, StructField, StringType

esquema_ptax_bruto = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", StringType(), True),
])

linhas_ptax = [
    (str(item.get("dataHoraCotacao")), str(item.get("cotacaoCompra")))
    for item in registros_ptax
]
df_ptax_bruto = spark.createDataFrame(linhas_ptax, schema=esquema_ptax_bruto)
df_ptax_bronze = df_ptax_bruto.withColumn(
    "ingestion_datetime", F.current_timestamp()
)

(df_ptax_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.tb_cotacao_dolar"))


## 7. Conferir as tabelas criadas


In [0]:
display(spark.sql("SHOW TABLES IN bronze"))
